# Q1: SMOTE-Style ROC Utility Benchmark

This notebook starts the Question 1 experiment from `manuscript/experiment_plan.md`:

> Can MIMIC match classical SMOTE-style ROC utility?

The reusable experiment machinery lives in `src/mimic_experiments/q1_smote_roc.py`. This notebook only chooses parameters, calls that module, and displays the resulting tables and plot.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "mimic").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import GenerationPolicy

from mimic_experiments.q1_smote_roc import (
    Q1Config,
    load_q1_dataset,
    manuscript_result_row,
    print_progress_event,
    q1_dataset_registry,
    roc_curve_points,
    run_mimic_roc_sweep,
)


## Experiment Controls

Edit this cell to choose the dataset, runtime profile, and MIMIC generation configuration. `run_profile="smoke"` keeps runtime modest while developing the pipeline; switch to `"paper"` for the full 10-fold protocol.


In [ ]:
# Dataset and run controls
DATASET = 0  # use a registry row number, or a key such as "adult_mixed"
RUN_PROFILE = "paper"  # "smoke" or "paper"
RANDOM_STATE = 0
N_ROWS_SMOKE = 250
N_JOBS = 1  # increase for parallel fold execution
ARTIFACT_DIR = str(PROJECT_ROOT / "manuscript" / "artifacts" / "q1_smote_roc")
CACHE_MODELS = True
WORKER_THREADS = 1  # keep BLAS/torch threads small when N_JOBS > 1

# MIMIC controls
MIMIC_MODE = "factorised"
MIMIC_CAPACITY_SMOKE = 0.0
MIMIC_CAPACITY_PAPER = 0.25

# MIMIC generation policy controls
POLICY_METHOD = "smote"
POLICY_NEIGHBOUR_MODE = "normal"
POLICY_N_NEIGHBORS = 5
POLICY_LAMBDA_RANGE = (0.0, 1.0)

# ROC sweep controls: fraction of the training-fold minority deficit to generate
DEFICIT_FRACTIONS = (0.0, 0.25, 0.5, 0.75, 1.0)

registry = q1_dataset_registry()
DATASET_KEY = registry.loc[DATASET, "key"] if isinstance(DATASET, int) else DATASET

config = Q1Config(
    dataset_key=DATASET_KEY,
    run_profile=RUN_PROFILE,
    random_state=RANDOM_STATE,
    n_rows_smoke=N_ROWS_SMOKE,
    n_jobs=N_JOBS,
    artifact_dir=ARTIFACT_DIR,
    cache_models=CACHE_MODELS,
    worker_threads=WORKER_THREADS,
    mimic_mode=MIMIC_MODE,
    mimic_capacity_smoke=MIMIC_CAPACITY_SMOKE,
    mimic_capacity_paper=MIMIC_CAPACITY_PAPER,
    deficit_fractions=DEFICIT_FRACTIONS,
    policy=GenerationPolicy(
        method=POLICY_METHOD,
        neighbour_mode=POLICY_NEIGHBOUR_MODE,
        n_neighbors=POLICY_N_NEIGHBORS,
        lambda_range=POLICY_LAMBDA_RANGE,
    ),
)
config


## Dataset Registry

The registry records the Q1 dataset distribution reported by the SMOTE paper and which loaders are available.


In [ ]:
display(q1_dataset_registry())


## Load Dataset


In [ ]:
df = load_q1_dataset(config)

display(df.head())
display(df["label"].value_counts().rename_axis("label").to_frame("count"))


## Run Q1 Sweep

Each deficit fraction gives one ROC point. MIMIC is fit only on the training fold, generated rows are appended only to that training fold, and the validation fold remains untouched.


In [ ]:
roc_points, q1_summary = run_mimic_roc_sweep(df, config, progress=print_progress_event)

display(q1_summary)
display(roc_points)


## Plot ROC Points


In [ ]:
curve = roc_curve_points(roc_points)

fig, ax = plt.subplots(figsize=(5.5, 5.0))
ax.plot(curve["mean_fpr"], curve["mean_tpr"], marker="o", linewidth=2, label="MIMIC ROC sweep")
ax.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.55)
for _, row in roc_points.iterrows():
    ax.annotate(f"{row['deficit_fraction']:.2g}", (row["mean_fpr"], row["mean_tpr"]), textcoords="offset points", xytext=(5, 5))
ax.set_title(f"Q1 MIMIC ROC sweep: {config.dataset_key}")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()


## Result Template For Manuscript

Fill the published-reference columns from the SMOTE paper once the corresponding dataset/classifier setup is selected.


In [ ]:
display(manuscript_result_row(config, q1_summary))
